In [1]:
# ---- Step 1: turn the raw QASM text into a clean list of lines ----
def clean_lines(text):
    lines = []
    for line in text.split("\n"):
        line = line.split("//")[0].strip()   # drop comments, drop whitespace
        if line:                              # skip blank lines
            lines.append(line)
    return lines


# ---- Step 2: read register sizes, e.g. "qreg q[3];" -> {"q": 3} ----
def get_registers(lines):
    registers = {}
    for line in lines:
        if line.startswith("qreg") or line.startswith("creg"):
            name = line.split()[1].split("[")[0]
            size = int(line.split("[")[1].split("]")[0])
            registers[name] = size
    return registers


# ---- Step 3: if there's a custom gate definition (like "gate syndrome ... {...}"),
#      pull out its name, its parameter names, and its body statements ----
def get_gate_definition(lines):
    for i, line in enumerate(lines):
        if line.startswith("gate "):
            header = line.split("{")[0]                  # "gate syndrome d1,d2,d3,a1,a2 "
            gate_name = header.split()[1]
            params = header.split()[2].split(",")

            body_text = ""
            for body_line in lines[i:]:
                body_text += body_line
                if "}" in body_line:
                    break
            body_text = body_text.split("{", 1)[1].split("}", 1)[0]
            body = [s.strip() for s in body_text.split(";") if s.strip()]
            return gate_name, params, body
    return None, None, None


# ---- Step 4: replace a call to the custom gate with its expanded statements,
#      substituting the call's real qubits for the definition's parameter names ----
def expand_gate_call(call_line, gate_name, params, body):
    args = call_line.split(gate_name, 1)[1].strip(" ;").split(",")
    args = [a.strip() for a in args]
    substitution = dict(zip(params, args))

    expanded = []
    for stmt in body:
        for param, real_qubit in substitution.items():
            stmt = stmt.replace(param, real_qubit)
        expanded.append(stmt)
    return expanded


# ---- Step 5: build one flat list of real statements: drop declarations,
#      inline any custom gate call, expand whole-register measures ----
def get_flat_statements(lines, registers):
    gate_name, params, body = get_gate_definition(lines)

    flat = []
    inside_gate_def = False
    for line in lines:
        if line.startswith("OPENQASM") or line.startswith("include"):
            continue
        if line.startswith("qreg") or line.startswith("creg"):
            continue
        if line.startswith("gate "):
            inside_gate_def = "}" not in line
            continue
        if inside_gate_def:
            if "}" in line:
                inside_gate_def = False
            continue

        for stmt in line.split(";"):
            stmt = stmt.strip()
            if not stmt:
                continue

            if gate_name and stmt.startswith(gate_name + " "):
                flat.extend(expand_gate_call(stmt, gate_name, params, body))
                continue

            if stmt.startswith("measure"):
                # "measure a -> syn"  ->  split into per-qubit lines if needed
                qubit_arg = stmt.split()[1]
                bit_arg = stmt.split("->")[1].strip()
                if "[" not in qubit_arg and qubit_arg in registers:
                    size = registers[qubit_arg]
                    for i in range(size):
                        flat.append(f"measure {qubit_arg}[{i}] -> {bit_arg}[{i}]")
                    continue

            flat.append(stmt)
    return flat


# ---- Step 6: find which classical registers get referenced by an "if" ----
def get_syndrome_registers(flat_statements):
    syndrome_registers = set()
    for stmt in flat_statements:
        if stmt.startswith("if("):
            reg_name = stmt.split("(")[1].split("==")[0].strip()
            syndrome_registers.add(reg_name)
    return syndrome_registers


# ---- Step 7: walk through and find stabilizer checks ----
def find_stabilizer_checks(flat_statements):
    syndrome_registers = get_syndrome_registers(flat_statements)

    pending_targets = {}   # ancilla qubit -> list of data qubits that cx'd into it
    checks = []
    check_num = 0

    for stmt in flat_statements:
        if stmt.startswith("cx"):
            qubits = stmt.replace("cx", "").strip().split(",")
            source, target = qubits[0].strip(), qubits[1].strip()
            pending_targets.setdefault(target, []).append(source)

        elif stmt.startswith("measure"):
            qubit = stmt.split()[1]
            bit = stmt.split("->")[1].strip()
            reg_name = bit.split("[")[0]

            if qubit in pending_targets and reg_name in syndrome_registers:
                checks.append({
                    "check_id": f"check_{check_num}",
                    "ancilla_qubit": qubit,
                    "data_qubits": pending_targets[qubit],
                    "syndrome_bit": bit,
                })
                check_num += 1
            pending_targets.pop(qubit, None)

    return checks


def extract_stabilizer_checks(qasm_text):
    lines = clean_lines(qasm_text)
    registers = get_registers(lines)
    flat_statements = get_flat_statements(lines, registers)
    return find_stabilizer_checks(flat_statements)

print("Parser loaded.")

Parser loaded.


In [2]:
import io, zipfile
from quantum_lake_student.config import Settings
from quantum_lake_student.connections import minio_client

settings = Settings.from_environment()
client = minio_client(settings)

key = "bronze/source=qasmbench/qasmbench-qec.zip"
response = client.get_object(settings.s3_bucket, key)
zip_bytes = response.read()
response.close()
response.release_conn()

circuits = [
    ("error_correctiond3_n5", "source", "small/error_correctiond3_n5/error_correctiond3_n5.qasm"),
    ("error_correctiond3_n5", "transpiled", "small/error_correctiond3_n5/error_correctiond3_n5_transpiled.qasm"),
    ("qec_en_n5", "source", "small/qec_en_n5/qec_en_n5.qasm"),
    ("qec_en_n5", "transpiled", "small/qec_en_n5/qec_en_n5_transpiled.qasm"),
    ("qec_sm_n5", "source", "small/qec_sm_n5/qec_sm_n5.qasm"),
    ("qec_sm_n5", "transpiled", "small/qec_sm_n5/qec_sm_n5_transpiled.qasm"),
]

stabilizer_check_rows = []

with zipfile.ZipFile(io.BytesIO(zip_bytes)) as z:
    for benchmark, variant, member in circuits:
        circuit_id = f"{benchmark}:{variant}"
        qasm_text = z.open(member).read().decode()
        checks = extract_stabilizer_checks(qasm_text)
        print(f"{circuit_id:35s} checks={len(checks)}")

        for check in checks:
            stabilizer_check_rows.append({
                "source_record_id": f"qasmbench:{member}:{check['check_id']}",
                "circuit_id": circuit_id,
                **check,
            })

error_correctiond3_n5:source        checks=0
error_correctiond3_n5:transpiled    checks=0
qec_en_n5:source                    checks=0
qec_en_n5:transpiled                checks=0
qec_sm_n5:source                    checks=2
qec_sm_n5:transpiled                checks=2


In [3]:
import pandas as pd
stabilizer_check_df = pd.DataFrame(stabilizer_check_rows)
display(stabilizer_check_df)

,source_record_id,circuit_id,check_id,ancilla_qubit,data_qubits,syndrome_bit
0,qasmbench:small/qec_sm_n5/qec_sm_n5.qasm:check_0,qec_sm_n5:source,check_0,a[0],"[q[0], q[1]]",syn[0]
1,qasmbench:small/qec_sm_n5/qec_sm_n5.qasm:check_1,qec_sm_n5:source,check_1,a[1],"[q[1], q[2]]",syn[1]
2,qasmbench:small/qec_sm_n5/qec_sm_n5_transpiled...,qec_sm_n5:transpiled,check_0,a[0],"[q[0], q[1]]",syn[0]
3,qasmbench:small/qec_sm_n5/qec_sm_n5_transpiled...,qec_sm_n5:transpiled,check_1,a[1],"[q[1], q[2]]",syn[1]


In [1]:
"""
Silver table 5 - stabilizer_check.parquet
STANDALONE notebook version. Run this in table_5.ipynb to test it end to
end. Once you're happy with it, the two functions in "table-building
functions" are written so they can be pasted straight into the Utility
class in prepare_data.py -- they're shaped exactly like
build_experiment_rows/google_experiment_table already are there:
  build_stabilizer_check_rows(...)  <-> build_experiment_rows(...)
  qasmbench_stabilizer_table(...)   <-> google_experiment_table(...)
Add "@staticmethod" and a "Utility." prefix on internal calls when you move
them in, same as the existing methods already do.
"""

import hashlib
import io
import re
import zipfile

import pandas as pd
from quantum_lake_student.config import Settings
from quantum_lake_student.connections import minio_client


# =========================================================================
# QASM parsing (your original table_5.ipynb functions, with the fixes we
# tested: statement-index tracking added, and expand_gate_call uses
# \bword\b substitution instead of plain .replace())
# =========================================================================

def clean_lines(text):
    lines = []
    for line in text.split("\n"):
        line = line.split("//")[0].strip()   # drop comments, drop whitespace
        if line:                              # skip blank lines
            lines.append(line)
    return lines


def get_registers(lines):
    registers = {}
    for line in lines:
        if line.startswith("qreg") or line.startswith("creg"):
            name = line.split()[1].split("[")[0]
            size = int(line.split("[")[1].split("]")[0])
            registers[name] = size
    return registers


def get_gate_definition(lines):
    for i, line in enumerate(lines):
        if line.startswith("gate "):
            header = line.split("{")[0]
            gate_name = header.split()[1]
            params = header.split()[2].split(",")

            body_text = ""
            for body_line in lines[i:]:
                body_text += body_line
                if "}" in body_line:
                    break
            body_text = body_text.split("{", 1)[1].split("}", 1)[0]
            body = [s.strip() for s in body_text.split(";") if s.strip()]
            return gate_name, params, body
    return None, None, None


def expand_gate_call(call_line, gate_name, params, body):
    args = call_line.split(gate_name, 1)[1].strip(" ;").split(",")
    args = [a.strip() for a in args]
    substitution = dict(zip(params, args))

    expanded = []
    for stmt in body:
        for param, real_qubit in substitution.items():
            stmt = re.sub(rf"\b{re.escape(param)}\b", real_qubit, stmt)
        expanded.append(stmt)
    return expanded


def get_flat_statements(lines, registers):
    """Same shape as your version, plus a source_stmt_index on every entry
    so each check can point back at exactly which statement produced it."""
    gate_name, params, body = get_gate_definition(lines)

    flat = []
    inside_gate_def = False
    stmt_index = 0
    for line in lines:
        if line.startswith("OPENQASM") or line.startswith("include"):
            stmt_index += 1
            continue
        if line.startswith("qreg") or line.startswith("creg"):
            stmt_index += 1
            continue
        if line.startswith("gate "):
            inside_gate_def = "}" not in line
            stmt_index += 1
            continue
        if inside_gate_def:
            if "}" in line:
                inside_gate_def = False
            continue
        if line.startswith("barrier"):
            continue  # a compiler fence, not an executed operation

        for stmt in line.split(";"):
            stmt = stmt.strip()
            if not stmt:
                continue

            this_index = stmt_index
            stmt_index += 1

            if gate_name and stmt.startswith(gate_name + " "):
                for expanded_stmt in expand_gate_call(stmt, gate_name, params, body):
                    flat.append((this_index, expanded_stmt))
                continue

            if stmt.startswith("measure"):
                qubit_arg = stmt.split()[1]
                bit_arg = stmt.split("->")[1].strip()
                if "[" not in qubit_arg and qubit_arg in registers:
                    size = registers[qubit_arg]
                    for i in range(size):
                        flat.append((this_index, f"measure {qubit_arg}[{i}] -> {bit_arg}[{i}]"))
                    continue

            flat.append((this_index, stmt))
    return flat


def get_syndrome_registers(flat_statements):
    syndrome_registers = set()
    for _idx, stmt in flat_statements:
        if stmt.startswith("if("):
            reg_name = stmt.split("(")[1].split("==")[0].strip()
            syndrome_registers.add(reg_name)
    return syndrome_registers


def find_stabilizer_checks(flat_statements):
    syndrome_registers = get_syndrome_registers(flat_statements)

    pending_targets = {}       # ancilla qubit -> (data qubits, build stmt indices)
    checks = []

    for idx, stmt in flat_statements:
        if stmt.startswith("cx"):
            qubits = stmt.replace("cx", "").strip().split(",")
            source, target = qubits[0].strip(), qubits[1].strip()
            data_qubits, build_indices = pending_targets.setdefault(target, ([], []))
            data_qubits.append(source)
            build_indices.append(idx)

        elif stmt.startswith("measure"):
            qubit = stmt.split()[1]
            bit = stmt.split("->")[1].strip()
            reg_name = bit.split("[")[0]

            if qubit in pending_targets and reg_name in syndrome_registers:
                data_qubits, build_indices = pending_targets[qubit]
                checks.append({
                    "check_id": qubit,
                    "ancilla_qubit": qubit,
                    "data_qubits": sorted(set(data_qubits)),
                    "syndrome_bit": bit,
                    "build_stmt_indices": sorted(set(build_indices)),
                    "measure_stmt_index": idx,
                })
            pending_targets.pop(qubit, None)

    return checks


def extract_stabilizer_checks(qasm_text):
    lines = clean_lines(qasm_text)
    registers = get_registers(lines)
    flat_statements = get_flat_statements(lines, registers)
    return find_stabilizer_checks(flat_statements), registers


def validate_check(check, registers):
    """Reject with a reason instead of silently accepting a malformed check."""
    problems = []

    def reg_and_index(qubit_ref):
        name, idx = qubit_ref.split("[")
        return name, int(idx.rstrip("]"))

    if not check["data_qubits"]:
        problems.append(("stabilizer_check_has_data_qubits", check["ancilla_qubit"]))
    if check["ancilla_qubit"] in check["data_qubits"]:
        problems.append(("stabilizer_ancilla_not_data_qubit", check["ancilla_qubit"]))

    for qubit_ref in [check["ancilla_qubit"], *check["data_qubits"]]:
        try:
            reg, idx = reg_and_index(qubit_ref)
        except ValueError:
            problems.append(("stabilizer_qubit_indexed", qubit_ref))
            continue
        if reg not in registers:
            problems.append(("stabilizer_register_declared", qubit_ref))
        elif idx >= registers[reg]:
            problems.append(("stabilizer_index_in_range", qubit_ref))

    syn_reg, syn_idx = check["syndrome_bit"].split("[")
    syn_idx = int(syn_idx.rstrip("]"))
    if syn_reg not in registers:
        problems.append(("stabilizer_syndrome_register_declared", check["syndrome_bit"]))
    elif syn_idx >= registers[syn_reg]:
        problems.append(("stabilizer_syndrome_bit_in_range", check["syndrome_bit"]))

    return problems


print("Parser loaded.")


# =========================================================================
# Table-building functions -- shaped to match build_experiment_rows /
# google_experiment_table already in prepare_data.py, so these paste in
# with minimal changes later.
# =========================================================================

QASMBENCH_KEY = "bronze/source=qasmbench/qasmbench-qec.zip"

QASMBENCH_CIRCUITS = [
    ("error_correctiond3_n5", "source", "small/error_correctiond3_n5/error_correctiond3_n5.qasm"),
    ("error_correctiond3_n5", "transpiled", "small/error_correctiond3_n5/error_correctiond3_n5_transpiled.qasm"),
    ("qec_en_n5", "source", "small/qec_en_n5/qec_en_n5.qasm"),
    ("qec_en_n5", "transpiled", "small/qec_en_n5/qec_en_n5_transpiled.qasm"),
    ("qec_sm_n5", "source", "small/qec_sm_n5/qec_sm_n5.qasm"),
    ("qec_sm_n5", "transpiled", "small/qec_sm_n5/qec_sm_n5_transpiled.qasm"),
]


def build_stabilizer_check_rows(raw_bytes, benchmark, variant, member, archive_sha256):
    """Mirrors build_experiment_rows(raw_bytes, folder): parses ONE circuit,
    returns (check_rows, trace_rows, issue_rows)."""
    circuit_id = f"qasmbench:{benchmark}:{variant}"  # matches table 6's convention

    with zipfile.ZipFile(io.BytesIO(raw_bytes)) as z:
        qasm_text = z.open(member).read().decode()

    checks, registers = extract_stabilizer_checks(qasm_text)

    check_rows, trace_rows, issue_rows = [], [], []
    for check in checks:
        source_record_id = (
            f"qasmbench:{member}#stmt={check['measure_stmt_index']}"
            f":check={check['ancilla_qubit']}"
        )
        problems = validate_check(check, registers)

        if problems:
            for rule_id, observed in problems:
                issue_rows.append({
                    "issue_id": "issue_" + hashlib.md5(
                        f"{source_record_id}:{rule_id}".encode()
                    ).hexdigest()[:12],
                    "source_record_id": source_record_id,
                    "rule_id": rule_id,
                    "severity": "error",
                    "observed_value": str(observed),
                    "action": "rejected",
                    "reason": rule_id.replace("_", " "),
                })
            continue

        check_rows.append({
            "source_record_id": source_record_id,
            "circuit_id": circuit_id,
            "check_id": check["check_id"],
            "ancilla_qubit": check["ancilla_qubit"],
            "data_qubits": check["data_qubits"],
            "syndrome_bit": check["syndrome_bit"],
        })

        for stmt_idx in sorted(set(check["build_stmt_indices"]) | {check["measure_stmt_index"]}):
            trace_rows.append({
                "source_record_id": source_record_id,
                "source_name": "qasmbench",
                "bronze_object": QASMBENCH_KEY,
                "archive_member": member,
                "record_locator": f"stmt={stmt_idx}",
                "input_sha256": archive_sha256,
            })

    return check_rows, trace_rows, issue_rows


def qasmbench_stabilizer_table(raw_bytes):
    """Mirrors google_experiment_table(raw_bytes): loops over every circuit,
    returns (check_df, trace_df, issues_df, candidate_count)."""
    archive_sha256 = hashlib.sha256(raw_bytes).hexdigest()

    all_checks, all_trace, all_issues = [], [], []
    candidate_count = 0

    for benchmark, variant, member in QASMBENCH_CIRCUITS:
        check_rows, trace_rows, issue_rows = build_stabilizer_check_rows(
            raw_bytes, benchmark, variant, member, archive_sha256
        )
        candidate_count += len(check_rows) + len(issue_rows)
        all_checks.extend(check_rows)
        all_trace.extend(trace_rows)
        all_issues.extend(issue_rows)

    check_df = pd.DataFrame(
        all_checks,
        columns=["source_record_id", "circuit_id", "check_id", "ancilla_qubit", "data_qubits", "syndrome_bit"],
    ).astype({
        "source_record_id": "string",
        "circuit_id": "string",
        "check_id": "string",
        "ancilla_qubit": "string",
        "syndrome_bit": "string",
    })
    trace_df = pd.DataFrame(all_trace)
    issues_df = pd.DataFrame(
        all_issues,
        columns=["issue_id", "source_record_id", "rule_id", "severity", "observed_value", "action", "reason"],
    )

    return check_df, trace_df, issues_df, candidate_count


# =========================================================================
# Standalone run -- fetch, parse, print. Nothing is written to silver/ or
# uploaded to MinIO here; this is just for you to eyeball the result before
# wiring it into prepare_data.py's run().
# =========================================================================

settings = Settings.from_environment()
client = minio_client(settings)

response = client.get_object(settings.s3_bucket, QASMBENCH_KEY)
raw_bytes = response.read()
response.close()
response.release_conn()

check_df, trace_df, issues_df, candidate_count = qasmbench_stabilizer_table(raw_bytes)

print(f"candidates={candidate_count}  accepted={len(check_df)}  "
      f"rejected={len(issues_df)}  trace_rows={len(trace_df)}")
print()
print(check_df.to_string(index=False))

Parser loaded.
candidates=4  accepted=4  rejected=0  trace_rows=10

                                                      source_record_id                     circuit_id check_id ancilla_qubit  data_qubits syndrome_bit
            qasmbench:small/qec_sm_n5/qec_sm_n5.qasm#stmt=9:check=a[0]     qasmbench:qec_sm_n5:source     a[0]          a[0] [q[0], q[1]]       syn[0]
            qasmbench:small/qec_sm_n5/qec_sm_n5.qasm#stmt=9:check=a[1]     qasmbench:qec_sm_n5:source     a[1]          a[1] [q[1], q[2]]       syn[1]
qasmbench:small/qec_sm_n5/qec_sm_n5_transpiled.qasm#stmt=11:check=a[0] qasmbench:qec_sm_n5:transpiled     a[0]          a[0] [q[0], q[1]]       syn[0]
qasmbench:small/qec_sm_n5/qec_sm_n5_transpiled.qasm#stmt=12:check=a[1] qasmbench:qec_sm_n5:transpiled     a[1]          a[1] [q[1], q[2]]       syn[1]
